In [15]:
# ── verification 
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# check all packages
packages = {
    'langchain':              'langchain',
    'chromadb':               'chromadb',
    'sentence_transformers':  'sentence_transformers',
    'faiss':                  'faiss',
    'transformers':           'transformers',
    'fastapi':                'fastapi',
}

print("package check:")
print("─" * 40)
all_good = True
for name, module in packages.items():
    try:
        __import__(module)
        print(f"  ✅ {name}")
    except ImportError:
        print(f"  ❌ {name} — run: pip install {name}")
        all_good = False

# check data
FHIR_DIR = r"C:\Users\nipas\Documents\clinical-llm-rag-pipeline\data\fhir\fhir"
json_files = list(Path(FHIR_DIR).glob("*.json"))
print()
print("data check:")
print("─" * 40)
print(f"  synthea files: {len(json_files):,}")

# check folders
folders = ['data/fhir/fhir','output/vectorstore',
           'output/charts','output/reports','app']
for f in folders:
    exists = Path(f).exists()
    print(f"  {'✅' if exists else '❌'} {f}")

print()
if all_good and len(json_files) > 0:
    print("✅ everything ready — start Cell 1!")
else:
    print("⚠️  fix issues above first")

package check:
────────────────────────────────────────
  ✅ langchain
  ✅ chromadb
  ✅ sentence_transformers
  ✅ faiss
  ✅ transformers
  ✅ fastapi

data check:
────────────────────────────────────────
  synthea files: 1,180
  ✅ data/fhir/fhir
  ✅ output/vectorstore
  ✅ output/charts
  ✅ output/reports
  ✅ app

✅ everything ready — start Cell 1!


# ============================================================
# CELL 1: Load FHIR Data & Extract Clinical Documents
# Project: Clinical LLM RAG Pipeline
# Standards: HL7 FHIR R4 · ClinicalBERT · LangChain · FAISS
# Author: Nipa Shah | github.com/nipa-analytics
# 
# Real-world parallel: COMPOSER-LLM at UCSD Health extracts
# patient data hourly via FHIR APIs and passes to LLM engine
# ============================================================

In [16]:
# ============================================================
# CELL 1 — FHIR R4 → Clinical Document Extraction
# ============================================================

import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────
FHIR_DIR = r"C:\Users\nipas\Documents\clinical-llm-rag-pipeline\data\fhir\fhir"

# ── LOAD FILES ─────────────────────────────────────────────
def load_bundles(fhir_dir, max_patients=200):
    bundles = []
    files = list(Path(fhir_dir).glob("*.json"))

    for f in files[:max_patients]:
        with open(f, "r", encoding="utf-8") as fp:
            bundles.append(json.load(fp))

    print(f"✅ Loaded {len(bundles)} FHIR bundles")
    return bundles


# ── EXTRACT CLINICAL DATA ─────────────────────────────────
def extract_clinical_documents(bundles):

    documents = []

    for bundle in bundles:

        patient_id = None
        gender = "unknown"
        city = ""
        state = ""
        birth_date = None

        conditions = []
        medications = []
        observations = []
        encounters = []
        procedures = []

        # ── parse bundle ───────────────────────────────
        for entry in bundle.get("entry", []):
            r = entry.get("resource", {})
            rtype = r.get("resourceType")

            # Patient
            if rtype == "Patient":
                patient_id = r.get("id")
                gender = r.get("gender", "unknown")
                birth_date = r.get("birthDate")

                addr = r.get("address", [{}])[0]
                city = addr.get("city", "")
                state = addr.get("state", "")

            # Condition
            elif rtype == "Condition":
                display = r.get("code", {}).get("text", "unknown")
                conditions.append(display)

            # Medication
            elif rtype == "MedicationRequest":
                med = r.get("medicationCodeableConcept", {}).get("text", "unknown")
                medications.append(med)

            # Observation
            elif rtype == "Observation":
                observations.append(r.get("code", {}).get("text", "observation"))

            # Encounter
            elif rtype == "Encounter":
                encounters.append(r.get("type", [{}])[0].get("text", "encounter"))

            # Procedure
            elif rtype == "Procedure":
                procedures.append(r.get("code", {}).get("text", "procedure"))

        # ── AGE CALCULATION ─────────────────────────────
        age = None
        if birth_date:
            try:
                dob = datetime.strptime(birth_date, "%Y-%m-%d")
                age = (datetime.now() - dob).days // 365
            except:
                age = None

        if not patient_id:
            continue

        # ── BUILD CLINICAL TEXT ─────────────────────────
        text = f"""
PATIENT CLINICAL SUMMARY
Patient ID: {patient_id}

Demographics: {gender}, Age {age or 'Unknown'}, {city}, {state}

ACTIVE CONDITIONS:
- {chr(10).join(conditions[:8])}

CURRENT MEDICATIONS:
- {chr(10).join(medications[:8])}

RECENT OBSERVATIONS:
- {chr(10).join(observations[:8])}

RECENT ENCOUNTERS:
- {chr(10).join(encounters[:3])}

RECENT PROCEDURES:
- {chr(10).join(procedures[:3])}
""".strip()

        documents.append({
            "patient_id": patient_id,
            "age": age,
            "gender": gender,
            "city": city,
            "state": state,
            "condition_count": len(conditions),
            "medication_count": len(medications),
            "clinical_text": text
        })

    print(f"✅ Extracted {len(documents)} clinical documents")
    return documents


# ── RUN CELL 1 ────────────────────────────────────────────
print("=" * 60)
print("CELL 1 — FHIR → Clinical Documents")
print("=" * 60)

bundles = load_bundles(FHIR_DIR, max_patients=200)
documents = extract_clinical_documents(bundles)

print("\nSample document:\n")
print(documents[0]["clinical_text"][:500])

CELL 1 — FHIR → Clinical Documents
✅ Loaded 200 FHIR bundles
✅ Extracted 200 clinical documents

Sample document:

PATIENT CLINICAL SUMMARY
Patient ID: 5cbc121b-cd71-4428-b8b7-31e53eba8184

Demographics: male, Age 80, Taunton, Massachusetts

ACTIVE CONDITIONS:
- Cardiac Arrest
History of cardiac arrest (situation)
Body mass index 30+ - obesity (finding)
Prediabetes
Anemia (disorder)
Polyp of colon
Viral sinusitis (disorder)
Viral sinusitis (disorder)

CURRENT MEDICATIONS:
- Ibuprofen 200 MG Oral Tablet

RECENT OBSERVATIONS:
- Body Height
Pain severity - 0-10 verbal numeric rating [Score] - Reported
Body Weig


In [17]:
import langchain
print(langchain.__version__)

1.3.11


In [18]:
%pip install -U langchain-text-splitters langchain-core langchain-community

Note: you may need to restart the kernel to use updated packages.


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

print("Everything works!")

Everything works!


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
import chromadb

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
client = chromadb.PersistentClient(path="output/vectorstore")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [22]:
import sys
print(sys.version)

3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


# ============================================================
# CELL 2: Text Chunking & Vector Embeddings
# 
# This is the CORE of any RAG pipeline.
# Real-world parallel: This is exactly what Abridge does —
# chunking physician notes into semantic segments, embedding
# them into vector space for instant clinical retrieval.
#
# Standards: LangChain · sentence-transformers · ChromaDB
# ============================================================

In [23]:
# ============================================================
# CELL 2 — Chunking + Embeddings + Vector Store
# CLEAN PRODUCTION VERSION
# ============================================================

import os
import numpy as np
import chromadb

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# ── safety check ───────────────────────────────────────────
if "documents" not in globals():
    raise Exception("Run Cell 1 first — documents not found")

# ── setup dirs ─────────────────────────────────────────────
os.makedirs("output/vectorstore", exist_ok=True)
os.makedirs("output/reports", exist_ok=True)

print("=" * 60)
print("CELL 2 — Chunking + Embeddings + Vector DB")
print("=" * 60)

# ── convert to LangChain docs ─────────────────────────────
lc_docs = [
    Document(page_content=d["clinical_text"], metadata=d)
    for d in documents
]

print(f"✅ LangChain docs: {len(lc_docs)}")

# ── chunking ──────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100
)

chunks = splitter.split_documents(lc_docs)

print(f"✅ chunks created: {len(chunks)}")

# ── embeddings model ──────────────────────────────────────
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ embedding model loaded")

# ── chromaDB ──────────────────────────────────────────────
client = chromadb.PersistentClient(
    path="output/vectorstore"
)

try:
    client.delete_collection("clinical_ehr")
except:
    pass

collection = client.create_collection(
    name="clinical_ehr",
    metadata={"hnsw:space": "cosine"}
)

# ── indexing ──────────────────────────────────────────────
BATCH = 50
total = 0

print("Indexing vectors...")

for i in range(0, len(chunks), BATCH):

    batch = chunks[i:i+BATCH]

    texts = [c.page_content for c in batch]
    metas = [c.metadata for c in batch]
    ids = [f"chunk_{i+j}" for j in range(len(batch))]

    emb = embedding_model.encode(texts).tolist()

    collection.add(
        documents=texts,
        embeddings=emb,
        metadatas=metas,
        ids=ids
    )

    total += len(batch)

print("\n✅ Vector DB ready")
print("Total vectors:", collection.count())

# ── test retrieval ────────────────────────────────────────
query = "diabetes patient medication"

q_emb = embedding_model.encode(query).tolist()

res = collection.query(
    query_embeddings=[q_emb],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)

print("\nTEST QUERY:", query)
print("Top result:")
print(res["documents"][0][0][:300])

CELL 2 — Chunking + Embeddings + Vector DB
✅ LangChain docs: 200
✅ chunks created: 447


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ embedding model loaded
Indexing vectors...

✅ Vector DB ready
Total vectors: 447

TEST QUERY: diabetes patient medication
Top result:
PATIENT CLINICAL SUMMARY
Patient ID: 3e35904e-5dc7-4324-9615-bb595f2b46e2

Demographics: male, Age 66, Boston, Massachusetts

ACTIVE CONDITIONS:
- Hypertension
Prediabetes
Diabetes
Hypertriglyceridemia (disorder)
Metabolic syndrome X (disorder)
Anemia (disorder)
Diabetic retinopathy associated with 


# ============================================================
# CELL 3 — Clinical RAG Engine (Retrieval + Answer Generation)
# ============================================================

In [24]:
import numpy as np
import json

from sentence_transformers import SentenceTransformer

# ── safety check ───────────────────────────────────────────
if "collection" not in globals():
    raise Exception("Run Cell 2 first — vector DB not found")

print("=" * 60)
print("CELL 3 — Clinical RAG Engine")
print("=" * 60)

# ── embedding model (same as Cell 2) ───────────────────────
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ── STEP 1: RETRIEVER FUNCTION ─────────────────────────────
def retrieve_context(query, k=3):

    q_emb = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]

    context_blocks = []

    for i in range(len(docs)):
        context_blocks.append({
            "text": docs[i],
            "metadata": metas[i],
            "score": float(1 - dists[i])  # cosine similarity
        })

    return context_blocks


# ── STEP 2: SIMPLE CLINICAL PROMPT BUILDER ────────────────
def build_prompt(query, contexts):

    context_text = "\n\n".join(
        [
            f"[Patient {c['metadata']['patient_id']} | "
            f"Age {c['metadata'].get('age','?')}]"
            f"\n{c['text']}"
            for c in contexts
        ]
    )

    prompt = f"""
You are a clinical AI assistant.

You MUST answer only using the provided clinical context.

If the answer is not in the context, say:
"Not enough clinical evidence in records."

---

CLINICAL CONTEXT:
{context_text}

---

QUESTION:
{query}

---

INSTRUCTIONS:
- Be precise
- Be medically safe
- Do NOT hallucinate
- Always refer to patient context
- Keep answer concise
"""

    return prompt


# ── STEP 3: SIMULATED LLM (SAFE FOR NOW) ──────────────────
def generate_answer(prompt, contexts):

    # NOTE: This is a placeholder.
    # Later we will replace with OpenAI / Llama / Med-PaLM style model.

    best_context = max(contexts, key=lambda x: x["score"])

    answer = {
        "answer": f"Based on clinical records, patient shows relevant findings in retrieved notes.",
        "supporting_patient_id": best_context["metadata"]["patient_id"],
        "confidence": round(best_context["score"], 3),
        "evidence_snippet": best_context["text"][:200]
    }

    return answer


# ── STEP 4: RAG PIPELINE FUNCTION ─────────────────────────
def rag_query(query):

    print("\n" + "-" * 60)
    print("QUESTION:", query)
    print("-" * 60)

    # retrieve
    contexts = retrieve_context(query, k=3)

    print(f"\nRetrieved {len(contexts)} contexts")

    for i, c in enumerate(contexts):
        print(f"\n[{i+1}] Patient:", c["metadata"]["patient_id"])
        print("Score:", round(c["score"], 4))
        print("Preview:", c["text"][:120], "...")

    # prompt
    prompt = build_prompt(query, contexts)

    # generate
    result = generate_answer(prompt, contexts)

    print("\n" + "-" * 60)
    print("FINAL ANSWER")
    print("-" * 60)

    print(json.dumps(result, indent=2))

    return result


# ── STEP 5: TEST QUERIES ──────────────────────────────────
test_queries = [
    "Does the patient have diabetes?",
    "What medications are prescribed?",
    "Any evidence of heart disease?",
]

for q in test_queries:
    rag_query(q)

CELL 3 — Clinical RAG Engine


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


------------------------------------------------------------
QUESTION: Does the patient have diabetes?
------------------------------------------------------------

Retrieved 3 contexts

[1] Patient: 80afed8b-2e43-48b9-bf2f-4a428d33af9a
Score: 0.5917
Preview: PATIENT CLINICAL SUMMARY
Patient ID: 80afed8b-2e43-48b9-bf2f-4a428d33af9a

Demographics: female, Age 105, Springfield, M ...

[2] Patient: 8850c4aa-cb77-4659-8373-980882405846
Score: 0.5773
Preview: PATIENT CLINICAL SUMMARY
Patient ID: 8850c4aa-cb77-4659-8373-980882405846

Demographics: male, Age 71, Marlborough, Mass ...

[3] Patient: 5afef64a-b152-4288-8cdd-9955d50fafb3
Score: 0.5444
Preview: PATIENT CLINICAL SUMMARY
Patient ID: 5afef64a-b152-4288-8cdd-9955d50fafb3

Demographics: female, Age 64, Everett, Massac ...

------------------------------------------------------------
FINAL ANSWER
------------------------------------------------------------
{
  "answer": "Based on clinical records, patient shows relevant findings in ret

# CELL 4 — Clinical Safety Layer (Hallucination Detector)

In [25]:
# ============================================================
# CELL 4 — Clinical Safety Layer (Hallucination Detection)
# ============================================================

import re
import numpy as np

print("=" * 60)
print("CELL 4 — Clinical Safety Layer")
print("=" * 60)

# ───────────────────────────────────────────────────────────
# STEP 1: Extract medical entities from text
# ───────────────────────────────────────────────────────────

def extract_medical_terms(text):
    """
    Lightweight clinical entity extractor.
    (In production: replace with MedSpaCy / SciSpacy / LLM NER)
    """

    patterns = {
        "disease": r"(diabetes|hypertension|heart failure|asthma|cancer|ckd|copd)",
        "medication": r"(metformin|insulin|lisinopril|atorvastatin|aspirin)",
        "lab": r"(glucose|hemoglobin|creatinine|cholesterol|bp|blood pressure)"
    }

    found = {}

    text_lower = text.lower()

    for k, pattern in patterns.items():
        matches = re.findall(pattern, text_lower)
        found[k] = list(set(matches))

    return found


# ───────────────────────────────────────────────────────────
# STEP 2: Evidence Checker
# ───────────────────────────────────────────────────────────

def check_evidence_alignment(question, contexts):
    """
    Checks whether the answer is supported by retrieved context.
    """

    question_terms = extract_medical_terms(question)

    context_text = " ".join([c["text"].lower() for c in contexts])
    context_terms = extract_medical_terms(context_text)

    missing_evidence = []

    for category in question_terms:
        for term in question_terms[category]:
            if term not in context_text:
                missing_evidence.append(term)

    confidence_score = 1 - (len(missing_evidence) /
                            max(1, sum(len(v) for v in question_terms.values())))

    return {
        "confidence": round(confidence_score, 3),
        "missing_evidence": missing_evidence,
        "context_entities": context_terms,
        "query_entities": question_terms
    }


# ───────────────────────────────────────────────────────────
# STEP 3: Hallucination Risk Scoring
# ───────────────────────────────────────────────────────────

def hallucination_risk_score(confidence):

    if confidence >= 0.8:
        return "LOW"
    elif confidence >= 0.5:
        return "MEDIUM"
    else:
        return "HIGH"


# ───────────────────────────────────────────────────────────
# STEP 4: Safety Wrapper for RAG output
# ───────────────────────────────────────────────────────────

def safety_check(query, contexts, rag_result):

    evidence = check_evidence_alignment(query, contexts)

    risk = hallucination_risk_score(evidence["confidence"])

    safe_output = {
        "original_answer": rag_result["answer"],
        "supporting_patient_id": rag_result["supporting_patient_id"],
        "confidence_score": evidence["confidence"],
        "hallucination_risk": risk,
        "missing_evidence": evidence["missing_evidence"],
        "retrieved_support_score": rag_result["confidence"],
        "evidence_snippet": rag_result["evidence_snippet"]
    }

    return safe_output


# ───────────────────────────────────────────────────────────
# STEP 5: TEST SAFETY LAYER
# (uses your Cell 3 rag_query function)
# ───────────────────────────────────────────────────────────

def safe_rag_query(query):

    print("\n" + "=" * 60)
    print("SAFE RAG QUERY:", query)
    print("=" * 60)

    # Step 1: retrieve
    contexts = retrieve_context(query, k=3)

    # Step 2: generate
    prompt = build_prompt(query, contexts)
    rag_result = generate_answer(prompt, contexts)

    # Step 3: safety check
    safe_result = safety_check(query, contexts, rag_result)

    print("\n🔍 SAFETY OUTPUT")
    print("-" * 40)

    import json
    print(json.dumps(safe_result, indent=2))

    return safe_result


# ───────────────────────────────────────────────────────────
# STEP 6: TEST CASES
# ───────────────────────────────────────────────────────────

test_queries = [
    "Does the patient have diabetes?",
    "Is there evidence of heart failure?",
    "Was insulin prescribed?",
]

for q in test_queries:
    safe_rag_query(q)

CELL 4 — Clinical Safety Layer

SAFE RAG QUERY: Does the patient have diabetes?

🔍 SAFETY OUTPUT
----------------------------------------
{
  "original_answer": "Based on clinical records, patient shows relevant findings in retrieved notes.",
  "supporting_patient_id": "80afed8b-2e43-48b9-bf2f-4a428d33af9a",
  "confidence_score": 1.0,
  "hallucination_risk": "LOW",
  "missing_evidence": [],
  "retrieved_support_score": 0.592,
  "evidence_snippet": "PATIENT CLINICAL SUMMARY\nPatient ID: 80afed8b-2e43-48b9-bf2f-4a428d33af9a\n\nDemographics: female, Age 105, Springfield, Massachusetts\n\nACTIVE CONDITIONS:\n- Prediabetes\nDiabetes\nNeuropathy due to type 2"
}

SAFE RAG QUERY: Is there evidence of heart failure?

🔍 SAFETY OUTPUT
----------------------------------------
{
  "original_answer": "Based on clinical records, patient shows relevant findings in retrieved notes.",
  "supporting_patient_id": "712d0848-2e66-44ae-95d1-2dc6f2c188cf",
  "confidence_score": 1.0,
  "hallucination_risk": "

# CELL 5 — RAG Evaluation Layer (Clinical QA Metrics)

In [26]:
# ============================================================
# CELL 5 — Clinical RAG Evaluation Layer
# (RAG Quality + Faithfulness + Retrieval Metrics)
# ============================================================

import numpy as np
import json

print("=" * 60)
print("CELL 5 — RAG Evaluation Layer")
print("=" * 60)

# ───────────────────────────────────────────────────────────
# STEP 1: Ground Truth Simulation (for demo)
# In real projects: use clinician-labeled dataset
# ───────────────────────────────────────────────────────────

ground_truth = {
    "Does the patient have diabetes?": "diabetes",
    "What medications are prescribed?": "medication",
    "Any evidence of heart disease?": "heart"
}

# ───────────────────────────────────────────────────────────
# STEP 2: Retrieval Quality Metric (Recall@K)
# ───────────────────────────────────────────────────────────

def recall_at_k(query, k=3):

    contexts = retrieve_context(query, k=k)

    retrieved_text = " ".join([c["text"].lower() for c in contexts])

    expected = ground_truth.get(query, "").lower()

    if expected == "":
        return 0.0

    return 1.0 if expected in retrieved_text else 0.0


# ───────────────────────────────────────────────────────────
# STEP 3: Context Precision (Simple Heuristic)
# ───────────────────────────────────────────────────────────

def context_precision(query, contexts):

    query_terms = query.lower().split()
    score = 0
    total = len(contexts)

    for c in contexts:
        text = c["text"].lower()
        if any(term in text for term in query_terms):
            score += 1

    return round(score / max(1, total), 3)


# ───────────────────────────────────────────────────────────
# STEP 4: Faithfulness Score (Answer vs Evidence)
# ───────────────────────────────────────────────────────────

def faithfulness_score(answer_text, contexts):

    answer_text = answer_text.lower()
    context_text = " ".join([c["text"].lower() for c in contexts])

    # overlap ratio between answer and retrieved evidence
    answer_terms = set(answer_text.split())
    context_terms = set(context_text.split())

    if len(answer_terms) == 0:
        return 0.0

    overlap = answer_terms.intersection(context_terms)

    return round(len(overlap) / len(answer_terms), 3)


# ───────────────────────────────────────────────────────────
# STEP 5: Full RAG Evaluation Pipeline
# ───────────────────────────────────────────────────────────

def evaluate_rag(query):

    print("\n" + "=" * 60)
    print("EVALUATING:", query)
    print("=" * 60)

    # retrieval
    contexts = retrieve_context(query, k=3)

    # generation
    prompt = build_prompt(query, contexts)
    rag_result = generate_answer(prompt, contexts)

    # safety layer (from Cell 4)
    safe_result = safety_check(query, contexts, rag_result)

    # metrics
    recall = recall_at_k(query, k=3)
    precision = context_precision(query, contexts)
    faithfulness = faithfulness_score(rag_result["answer"], contexts)

    report = {
        "query": query,
        "recall@3": recall,
        "context_precision": precision,
        "faithfulness": faithfulness,
        "hallucination_risk": safe_result["hallucination_risk"],
        "confidence_score": safe_result["confidence_score"],
        "supporting_patient_id": safe_result["supporting_patient_id"]
    }

    print("\n📊 METRICS REPORT")
    print("-" * 40)
    print(json.dumps(report, indent=2))

    return report


# ───────────────────────────────────────────────────────────
# STEP 6: RUN EVALUATION TEST SET
# ───────────────────────────────────────────────────────────

test_set = [
    "Does the patient have diabetes?",
    "What medications are prescribed?",
    "Any evidence of heart disease?"
]

results = []

for q in test_set:
    results.append(evaluate_rag(q))

# ───────────────────────────────────────────────────────────
# STEP 7: SUMMARY STATISTICS
# ───────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("OVERALL SYSTEM PERFORMANCE")
print("=" * 60)

avg_recall = np.mean([r["recall@3"] for r in results])
avg_precision = np.mean([r["context_precision"] for r in results])
avg_faithfulness = np.mean([r["faithfulness"] for r in results])

print(f"Recall@3:        {avg_recall:.3f}")
print(f"Context Precision:{avg_precision:.3f}")
print(f"Faithfulness:     {avg_faithfulness:.3f}")

print("\n✅ RAG evaluation complete")

CELL 5 — RAG Evaluation Layer

EVALUATING: Does the patient have diabetes?

📊 METRICS REPORT
----------------------------------------
{
  "query": "Does the patient have diabetes?",
  "recall@3": 1.0,
  "context_precision": 1.0,
  "faithfulness": 0.182,
  "hallucination_risk": "LOW",
  "confidence_score": 1.0,
  "supporting_patient_id": "80afed8b-2e43-48b9-bf2f-4a428d33af9a"
}

EVALUATING: What medications are prescribed?

📊 METRICS REPORT
----------------------------------------
{
  "query": "What medications are prescribed?",
  "recall@3": 1.0,
  "context_precision": 1.0,
  "faithfulness": 0.0,
  "hallucination_risk": "LOW",
  "confidence_score": 1.0,
  "supporting_patient_id": "822f82f3-3d03-4ce1-8c2b-14764c244fcf"
}

EVALUATING: Any evidence of heart disease?

📊 METRICS REPORT
----------------------------------------
{
  "query": "Any evidence of heart disease?",
  "recall@3": 0.0,
  "context_precision": 0.333,
  "faithfulness": 0.182,
  "hallucination_risk": "LOW",
  "confidence_s

# CELL 6 — FastAPI Clinical RAG Service (PRODUCTION STYLE)

In [27]:
# ============================================================
# CELL 6 — FastAPI Clinical RAG API
# Production-style inference layer for your RAG system
# ============================================================

from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

print("=" * 60)
print("CELL 6 — FastAPI Clinical RAG API")
print("=" * 60)

# ───────────────────────────────────────────────────────────
# STEP 1: Initialize FastAPI app
# ───────────────────────────────────────────────────────────

app = FastAPI(
    title="Clinical RAG Pipeline API",
    description="FHIR + RAG + Safety Layer for Clinical QA",
    version="1.0"
)

# ───────────────────────────────────────────────────────────
# STEP 2: Request Schema
# ───────────────────────────────────────────────────────────

class QueryRequest(BaseModel):
    question: str


# ───────────────────────────────────────────────────────────
# STEP 3: Response Schema
# ───────────────────────────────────────────────────────────

class QueryResponse(BaseModel):
    question: str
    answer: str
    patient_id: str
    confidence: float
    hallucination_risk: str
    evidence_snippet: str


# ───────────────────────────────────────────────────────────
# STEP 4: Core RAG Pipeline Wrapper
# (uses Cell 3 + Cell 4 functions)
# ───────────────────────────────────────────────────────────

def run_rag_pipeline(question: str):

    # retrieval
    contexts = retrieve_context(question, k=3)

    # generation
    prompt = build_prompt(question, contexts)
    rag_result = generate_answer(prompt, contexts)

    # safety layer
    safe_result = safety_check(question, contexts, rag_result)

    return contexts, rag_result, safe_result


# ───────────────────────────────────────────────────────────
# STEP 5: API Endpoint — Clinical Q&A
# ───────────────────────────────────────────────────────────

@app.post("/query", response_model=QueryResponse)
def query(request: QueryRequest):

    contexts, rag_result, safe_result = run_rag_pipeline(request.question)

    return QueryResponse(
        question=request.question,
        answer=rag_result["answer"],
        patient_id=safe_result["supporting_patient_id"],
        confidence=safe_result["confidence_score"],
        hallucination_risk=safe_result["hallucination_risk"],
        evidence_snippet=rag_result["evidence_snippet"]
    )


# ───────────────────────────────────────────────────────────
# STEP 6: Health Check Endpoint
# ───────────────────────────────────────────────────────────

@app.get("/health")
def health():
    return {
        "status": "ok",
        "message": "Clinical RAG API is running"
    }


# ───────────────────────────────────────────────────────────
# STEP 7: Local Run (Jupyter-safe version)
# ───────────────────────────────────────────────────────────

print("\n🚀 API is defined successfully!")
print("To run it, execute this in terminal (NOT notebook):")
print()
print("uvicorn cell6_api:app --reload")
print()
print("Or save this as api.py and run:")
print("uvicorn api:app --reload")

CELL 6 — FastAPI Clinical RAG API

🚀 API is defined successfully!
To run it, execute this in terminal (NOT notebook):

uvicorn cell6_api:app --reload

Or save this as api.py and run:
uvicorn api:app --reload


# Functional-testing 

In [14]:
run_rag_pipeline("Does the patient have diabetes?")

([{'text': 'PATIENT CLINICAL SUMMARY\nPatient ID: 80afed8b-2e43-48b9-bf2f-4a428d33af9a\n\nDemographics: female, Age 105, Springfield, Massachusetts\n\nACTIVE CONDITIONS:\n- Prediabetes\nDiabetes\nNeuropathy due to type 2 diabetes mellitus (disorder)\nHypertriglyceridemia (disorder)\nMetabolic syndrome X (disorder)\nDiabetic retinopathy associated with type II diabetes mellitus (disorder)\nHyperlipidemia\nHyperglycemia (disorder)',
   'metadata': {'gender': 'female',
    'city': 'Springfield',
    'condition_count': 16,
    'age': 105,
    'state': 'Massachusetts',
    'patient_id': '80afed8b-2e43-48b9-bf2f-4a428d33af9a',
    'clinical_text': 'PATIENT CLINICAL SUMMARY\nPatient ID: 80afed8b-2e43-48b9-bf2f-4a428d33af9a\n\nDemographics: female, Age 105, Springfield, Massachusetts\n\nACTIVE CONDITIONS:\n- Prediabetes\nDiabetes\nNeuropathy due to type 2 diabetes mellitus (disorder)\nHypertriglyceridemia (disorder)\nMetabolic syndrome X (disorder)\nDiabetic retinopathy associated with type II